# Vacancies 2026 — Salary Prediction v6
**Key insight:** `MAPE(employer_id, experience) median` = **0.1165** on train

**Pipeline:** Hierarchical lookup + LGB+XGB blend, alpha weighted by lookup confidence level

### 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

SEED=42; N_SPLITS=5; STE_K=20
np.random.seed(SEED)

### 2. Data Loading

In [2]:
train=pd.read_csv('data/train.csv')
test=pd.read_csv('data/test_x.csv')
print(f'train: {train.shape}, test: {test.shape}')

train: (49051, 26), test: (12263, 25)


### 3. Hierarchical Lookup Table

In [3]:
TARGET='salary_mean_net'
y_raw=train[TARGET].values.astype(np.float64)

lv1=train.groupby(['employer_id','experience_name'])[TARGET].agg(median='median',count='count').reset_index()
lv2=train.groupby('employer_id')[TARGET].agg(median='median',count='count').reset_index()
lv3=train.groupby(['professional_roles_name','experience_name'])[TARGET].agg(median='median',count='count').reset_index()
lv4=train.groupby('professional_roles_name')[TARGET].agg(median='median',count='count').reset_index()
lv5=train.groupby(['specializations_profarea_name','experience_name'])[TARGET].agg(median='median',count='count').reset_index()
global_median=train[TARGET].median()

print(f'LV1:{len(lv1)} LV2:{len(lv2)} LV3:{len(lv3)} LV4:{len(lv4)} global:{global_median:.0f}')

LV1:27583 LV2:23925 LV3:508 LV4:145 global:42920


In [4]:
def hierarchical_lookup(df):
    r=df[['employer_id','experience_name','professional_roles_name','specializations_profarea_name']].copy()
    r['lookup_pred']=np.nan; r['lookup_count']=0; r['lookup_level']=6

    m=r.merge(lv1[['employer_id','experience_name','median','count']],on=['employer_id','experience_name'],how='left')
    mk=m['median'].notna()
    r.loc[mk,'lookup_pred']=m.loc[mk,'median'].values; r.loc[mk,'lookup_count']=m.loc[mk,'count'].values; r.loc[mk,'lookup_level']=1

    need=r['lookup_pred'].isna()
    m=r[need].merge(lv2[['employer_id','median','count']],on='employer_id',how='left')
    mk=m['median'].notna(); idx=r[need].index[mk]
    r.loc[idx,'lookup_pred']=m.loc[mk,'median'].values; r.loc[idx,'lookup_count']=m.loc[mk,'count'].values; r.loc[idx,'lookup_level']=2

    need=r['lookup_pred'].isna()
    m=r[need].merge(lv3[['professional_roles_name','experience_name','median','count']],on=['professional_roles_name','experience_name'],how='left')
    mk=m['median'].notna(); idx=r[need].index[mk]
    r.loc[idx,'lookup_pred']=m.loc[mk,'median'].values; r.loc[idx,'lookup_count']=m.loc[mk,'count'].values; r.loc[idx,'lookup_level']=3

    need=r['lookup_pred'].isna()
    m=r[need].merge(lv4[['professional_roles_name','median','count']],on='professional_roles_name',how='left')
    mk=m['median'].notna(); idx=r[need].index[mk]
    r.loc[idx,'lookup_pred']=m.loc[mk,'median'].values; r.loc[idx,'lookup_count']=m.loc[mk,'count'].values; r.loc[idx,'lookup_level']=4

    need=r['lookup_pred'].isna()
    m=r[need].merge(lv5[['specializations_profarea_name','experience_name','median','count']],on=['specializations_profarea_name','experience_name'],how='left')
    mk=m['median'].notna(); idx=r[need].index[mk]
    r.loc[idx,'lookup_pred']=m.loc[mk,'median'].values; r.loc[idx,'lookup_count']=m.loc[mk,'count'].values; r.loc[idx,'lookup_level']=5

    na=r['lookup_pred'].isna()
    r.loc[na,'lookup_pred']=global_median; r.loc[na,'lookup_count']=1; r.loc[na,'lookup_level']=6
    return r[['lookup_pred','lookup_count','lookup_level']]

train_lookup=hierarchical_lookup(train)
mape_in=np.mean(np.abs((y_raw-train_lookup['lookup_pred'].values)/(y_raw+1e-8)))
print(f'Train lookup MAPE (in-sample): {mape_in:.4f}')
print(train_lookup['lookup_level'].value_counts().sort_index())
test_lookup=hierarchical_lookup(test)
print('\nTest lookup level dist:')
print(test_lookup['lookup_level'].value_counts().sort_index())

Train lookup MAPE (in-sample): 0.1167
lookup_level
1    49008
3       43
Name: count, dtype: int64

Test lookup level dist:
lookup_level
1    6557
2    1085
3    4614
4       7
Name: count, dtype: int64


### 4. OOF Lookup MAPE (unbiased estimate)

In [5]:
kf=KFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
oof_lookup=np.zeros(len(train))

for tr_idx,val_idx in kf.split(train):
    tr_f=train.iloc[tr_idx]; val_f=train.iloc[val_idx]
    l1=tr_f.groupby(['employer_id','experience_name'])[TARGET].median()
    l2=tr_f.groupby('employer_id')[TARGET].median()
    l3=tr_f.groupby(['professional_roles_name','experience_name'])[TARGET].median()
    l4=tr_f.groupby('professional_roles_name')[TARGET].median()
    l5=tr_f.groupby(['specializations_profarea_name','experience_name'])[TARGET].median()
    gm=tr_f[TARGET].median()
    preds=np.full(len(val_f),gm)
    for i,(_,row) in enumerate(val_f.iterrows()):
        k1=(row['employer_id'],row['experience_name']); k2=row['employer_id']
        k3=(row['professional_roles_name'],row['experience_name']); k4=row['professional_roles_name']
        k5=(row['specializations_profarea_name'],row['experience_name'])
        if k1 in l1.index: preds[i]=l1[k1]
        elif k2 in l2.index: preds[i]=l2[k2]
        elif k3 in l3.index: preds[i]=l3[k3]
        elif k4 in l4.index: preds[i]=l4[k4]
        elif k5 in l5.index: preds[i]=l5[k5]
    oof_lookup[val_idx]=preds

oof_lookup_mape=np.mean(np.abs((y_raw-oof_lookup)/(y_raw+1e-8)))
print(f'OOF Lookup MAPE (unbiased): {oof_lookup_mape:.4f}')

OOF Lookup MAPE (unbiased): 0.3460


### 5. Feature Engineering

In [6]:
DESC_COL='lemmaized_wo_stopwords_raw_description'
TITLE_COL='name_clean'
for df in [train,test]:
    df[DESC_COL]=df[DESC_COL].fillna('')
    df[TITLE_COL]=df[TITLE_COL].fillna('')

n_train=len(train)
corp_d=pd.concat([train[DESC_COL],test[DESC_COL]],ignore_index=True)
corp_t=pd.concat([train[TITLE_COL],test[TITLE_COL]],ignore_index=True)

tfd=TfidfVectorizer(max_features=6000,sublinear_tf=True,min_df=3,ngram_range=(1,2),dtype=np.float32)
tft=TfidfVectorizer(max_features=3000,sublinear_tf=True,min_df=2,ngram_range=(1,2),dtype=np.float32)
svd_d=TruncatedSVD(n_components=100,random_state=SEED)
svd_t=TruncatedSVD(n_components=40,random_state=SEED)

md_=svd_d.fit_transform(tfd.fit_transform(corp_d)).astype(np.float32)
mt_=svd_t.fit_transform(tft.fit_transform(corp_t)).astype(np.float32)

cols_svd=[f'desc_svd_{i}' for i in range(100)]+[f'title_svd_{i}' for i in range(40)]
SVD_TRAIN=pd.DataFrame(np.hstack([md_[:n_train],mt_[:n_train]]),columns=cols_svd)
SVD_TEST=pd.DataFrame(np.hstack([md_[n_train:],mt_[n_train:]]),columns=cols_svd)
print(f'SVD ready: {SVD_TRAIN.shape}')

SVD ready: (49051, 140)


In [7]:
EXP_ORD=['Нет опыта','От 1 года до 3 лет','От 3 до 6 лет','Более 6 лет']
DROP_COLS=['id','salary_mean_net','raw_description','raw_branded_description',
           'lemmaized_wo_stopwords_raw_branded_description','lemmaized_wo_stopwords_raw_description',
           'name','name_clean','unified_address_country']
LOW_CARD=['schedule_name','employment_name','is_branded_description',
          'if_foreign_language','accept_handicapped','accept_kids']
TE_COLS=['employer_id','employer_name','professional_roles_name',
         'specializations_profarea_name','employer_industries',
         'unified_address_city','unified_address_state','unified_address_region']

exp_map={v:i for i,v in enumerate(EXP_ORD)}
for df in [train,test]:
    df['experience_ord']=df['experience_name'].map(exp_map).fillna(-1).astype(np.int8)
    df['desc_word_count']=df[DESC_COL].str.split().str.len().fillna(0).astype(np.int16)
    df['has_skills']=(df['key_skills_name'].fillna('[]')!='[]').astype(np.int8)
    df['has_languages']=(df['languages_name'].fillna('[]')!='[]').astype(np.int8)
    df['skills_count']=df['key_skills_name'].fillna('[]').str.count(',').astype(np.int16)
    city=df['unified_address_city'].fillna('').str.lower()
    df['is_moscow']=city.str.contains('москва').astype(np.int8)
    df['is_spb']=city.str.contains('санкт').astype(np.int8)
    for col in TE_COLS:
        if col in df.columns:
            df[col]=df[col].fillna('__NA__').astype(str)

oe=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1,dtype=np.float32)
train[LOW_CARD]=oe.fit_transform(train[LOW_CARD].astype(str))
test[LOW_CARD]=oe.transform(test[LOW_CARD].astype(str))

train['lookup_pred']=oof_lookup
train['lookup_level']=train_lookup['lookup_level'].values
test['lookup_pred']=test_lookup['lookup_pred'].values
test['lookup_level']=test_lookup['lookup_level'].values

y=np.log1p(y_raw).astype(np.float32)
gm=float(np.mean(y))

base_cols=[c for c in train.columns
           if c not in DROP_COLS+TE_COLS+['experience_name','key_skills_name','languages_name']
           and train[c].dtype!=object]

X_base_tr=train[base_cols].reset_index(drop=True)
X_base_te=test[[c for c in base_cols if c in test.columns]].reset_index(drop=True)
X_base_te=X_base_te.reindex(columns=X_base_tr.columns)
print(f'Base features: {X_base_tr.shape[1]}')

Base features: 15


In [8]:
def smoothed_te(col_tr,col_val,col_te,y_tr,global_mean,k=STE_K):
    stats=pd.DataFrame({'y':y_tr,'cat':col_tr.values}).groupby('cat')['y'].agg(['mean','count'])
    stats['smooth']=(stats['count']*stats['mean']+k*global_mean)/(stats['count']+k)
    te_map=stats['smooth']
    return col_tr.map(te_map).fillna(global_mean),col_val.map(te_map).fillna(global_mean),col_te.map(te_map).fillna(global_mean)

def build_fold(tr_idx,val_idx):
    tl,vl,el=[],[],[]
    for col in TE_COLS:
        s_tr,s_val,s_te=smoothed_te(
            train[col].iloc[tr_idx].reset_index(drop=True),
            train[col].iloc[val_idx].reset_index(drop=True),
            test[col].reset_index(drop=True),y[tr_idx],gm)
        tl.append(s_tr.rename(f'te_{col}')); vl.append(s_val.rename(f'te_{col}')); el.append(s_te.rename(f'te_{col}'))
    te_tr=pd.concat(tl,axis=1); te_val=pd.concat(vl,axis=1); te_te=pd.concat(el,axis=1)
    def asm(base,svd,te):
        out=pd.concat([base,svd,te],axis=1)
        out['exp_x_role_te']=out['experience_ord'].astype(float)*out['te_professional_roles_name']
        out['moscow_x_exp']=out['is_moscow'].astype(float)*out['experience_ord'].astype(float)
        return out
    return (asm(X_base_tr.iloc[tr_idx].reset_index(drop=True),SVD_TRAIN.iloc[tr_idx].reset_index(drop=True),te_tr),
            asm(X_base_tr.iloc[val_idx].reset_index(drop=True),SVD_TRAIN.iloc[val_idx].reset_index(drop=True),te_val),
            asm(X_base_te.reset_index(drop=True),SVD_TEST.reset_index(drop=True),te_te))

print('Fold helper ready.')

Fold helper ready.


### 6. LightGBM OOF

In [9]:
lgb_params={'objective':'regression','metric':'rmse','n_estimators':8000,'learning_rate':0.02,
            'num_leaves':255,'min_child_samples':15,'feature_fraction':0.6,
            'bagging_fraction':0.8,'bagging_freq':5,'reg_alpha':0.05,'reg_lambda':0.1,
            'random_state':SEED,'n_jobs':-1,'verbose':-1}

oof_lgb=np.zeros(len(train),dtype=np.float64)
pred_lgb=np.zeros(len(test),dtype=np.float64)

for fold,(tr_idx,val_idx) in enumerate(kf.split(X_base_tr)):
    X_tr,X_val,X_te=build_fold(tr_idx,val_idx)
    m=lgb.LGBMRegressor(**lgb_params)
    m.fit(X_tr,y[tr_idx],eval_set=[(X_val,y[val_idx])],
          callbacks=[lgb.early_stopping(200,verbose=False),lgb.log_evaluation(1000)])
    vp=np.expm1(m.predict(X_val))
    oof_lgb[val_idx]=vp
    pred_lgb+=np.expm1(m.predict(X_te))/N_SPLITS
    mape=np.mean(np.abs((y_raw[val_idx]-vp)/(y_raw[val_idx]+1e-8)))
    print(f'Fold {fold+1} | LGB MAPE: {mape:.4f} | iter: {m.best_iteration_}')

lgb_oof_mape=np.mean(np.abs((y_raw-oof_lgb)/(y_raw+1e-8)))
print(f'\nLightGBM OOF MAPE: {lgb_oof_mape:.4f}')

[1000]	valid_0's rmse: 0.380715
[2000]	valid_0's rmse: 0.380179
Fold 1 | LGB MAPE: 0.2778 | iter: 2236
[1000]	valid_0's rmse: 0.370248
[2000]	valid_0's rmse: 0.369389
[3000]	valid_0's rmse: 0.36928
Fold 2 | LGB MAPE: 0.2695 | iter: 3331
[1000]	valid_0's rmse: 0.375373
[2000]	valid_0's rmse: 0.374598
Fold 3 | LGB MAPE: 0.2750 | iter: 2492
[1000]	valid_0's rmse: 0.370898
[2000]	valid_0's rmse: 0.370409
Fold 4 | LGB MAPE: 0.2706 | iter: 1900
[1000]	valid_0's rmse: 0.370265
[2000]	valid_0's rmse: 0.369652
Fold 5 | LGB MAPE: 0.2710 | iter: 2364

LightGBM OOF MAPE: 0.2728


### 7. XGBoost OOF

In [10]:
xgb_params={'objective':'reg:squarederror','eval_metric':'rmse','n_estimators':8000,
            'learning_rate':0.02,'max_depth':7,'min_child_weight':10,
            'subsample':0.8,'colsample_bytree':0.6,'reg_alpha':0.05,'reg_lambda':0.1,
            'random_state':SEED,'n_jobs':-1,'tree_method':'hist',
            'early_stopping_rounds':200,'verbosity':0}

oof_xgb=np.zeros(len(train),dtype=np.float64)
pred_xgb=np.zeros(len(test),dtype=np.float64)

for fold,(tr_idx,val_idx) in enumerate(kf.split(X_base_tr)):
    X_tr,X_val,X_te=build_fold(tr_idx,val_idx)
    m=xgb.XGBRegressor(**xgb_params)
    m.fit(X_tr,y[tr_idx],eval_set=[(X_val,y[val_idx])],verbose=False)
    vp=np.expm1(m.predict(X_val))
    oof_xgb[val_idx]=vp
    pred_xgb+=np.expm1(m.predict(X_te))/N_SPLITS
    mape=np.mean(np.abs((y_raw[val_idx]-vp)/(y_raw[val_idx]+1e-8)))
    print(f'Fold {fold+1} | XGB MAPE: {mape:.4f} | iter: {m.best_iteration}')

xgb_oof_mape=np.mean(np.abs((y_raw-oof_xgb)/(y_raw+1e-8)))
print(f'\nXGBoost OOF MAPE: {xgb_oof_mape:.4f}')

Fold 1 | XGB MAPE: 0.2739 | iter: 4180
Fold 2 | XGB MAPE: 0.2634 | iter: 7329
Fold 3 | XGB MAPE: 0.2715 | iter: 3691
Fold 4 | XGB MAPE: 0.2638 | iter: 5202
Fold 5 | XGB MAPE: 0.2651 | iter: 5682

XGBoost OOF MAPE: 0.2675


### 8. Ensemble Blending

In [11]:
oof_ml=0.5*oof_lgb+0.5*oof_xgb
pred_ml=0.5*pred_lgb+0.5*pred_xgb
ml_oof_mape=np.mean(np.abs((y_raw-oof_ml)/(y_raw+1e-8)))
print(f'ML ensemble OOF MAPE: {ml_oof_mape:.4f}')
print(f'Lookup OOF MAPE:       {oof_lookup_mape:.4f}')

# Lookup alpha by level: higher confidence = higher alpha
ALPHA_MAP={1:0.80,2:0.60,3:0.30,4:0.15,5:0.10,6:0.05}

train_levels=train_lookup['lookup_level'].values
alpha_oof=np.array([ALPHA_MAP[int(lv)] for lv in train_levels])
oof_blend=alpha_oof*oof_lookup+(1-alpha_oof)*oof_ml
blend_oof_mape=np.mean(np.abs((y_raw-oof_blend)/(y_raw+1e-8)))
print(f'\nBlend OOF MAPE: {blend_oof_mape:.4f}')

print('\nPer-level MAPE breakdown:')
for lv in sorted(ALPHA_MAP.keys()):
    mask=train_levels==lv
    if mask.sum()>0:
        m_lk=np.mean(np.abs((y_raw[mask]-oof_lookup[mask])/(y_raw[mask]+1e-8)))
        m_ml=np.mean(np.abs((y_raw[mask]-oof_ml[mask])/(y_raw[mask]+1e-8)))
        m_bl=np.mean(np.abs((y_raw[mask]-oof_blend[mask])/(y_raw[mask]+1e-8)))
        print(f'  LV{lv} (n={mask.sum():5d}, alpha={ALPHA_MAP[lv]:.2f}): lookup={m_lk:.4f} ml={m_ml:.4f} blend={m_bl:.4f}')

ML ensemble OOF MAPE: 0.2695
Lookup OOF MAPE:       0.3460

Blend OOF MAPE: 0.3211

Per-level MAPE breakdown:
  LV1 (n=49008, alpha=0.80): lookup=0.3460 ml=0.2695 blend=0.3212
  LV3 (n=   43, alpha=0.30): lookup=0.3573 ml=0.2651 blend=0.2762


### 9. Submission

In [12]:
# Final test predictions
test_levels=test_lookup['lookup_level'].values
alpha_test=np.array([ALPHA_MAP[int(lv)] for lv in test_levels])
pred_blend=alpha_test*test_lookup['lookup_pred'].values+(1-alpha_test)*pred_ml

sub=pd.DataFrame({'id':test['id'],'salary_mean_net':pred_blend})
sub.to_csv('submission_v6.csv',index=False)
print(f'Saved submission_v6.csv  shape={sub.shape}')
print(sub['salary_mean_net'].describe())
print('\nTest level distribution:')
print(pd.Series(test_levels).value_counts().sort_index())

Saved submission_v6.csv  shape=(12263, 2)
count     12263.000000
mean      47688.714175
std       19877.268957
min       13612.454086
25%       36409.622780
50%       43467.486192
75%       53229.165494
max      179902.726369
Name: salary_mean_net, dtype: float64

Test level distribution:
1    6557
2    1085
3    4614
4       7
Name: count, dtype: int64
